# Clarity: LoRA fine-tune of Qwen3-4B-Instruct-2507 → `clarity-simplify` (Prompt 10)

Free Colab **T4** notebook. Run the cells top to bottom (**Runtime → Run all** works). If Colab disconnects, reconnect and run all again: training resumes from the last checkpoint on Google Drive.

Before you start (see `training/README.md`):
1. **Runtime → Change runtime type → T4 GPU.**
2. Upload `sft_train.jsonl` and `sft_dev.jsonl` to **`MyDrive/clarity/data/`** in Google Drive.

What it does: 4-bit QLoRA (r 16, α 16) on all attention and MLP projections, 1 epoch, loss only on the assistant answer, dev loss every 100 steps, checkpoints to Drive every 100 steps, then a GGUF `q4_k_m` export for Ollama.

## 1. Install (pinned) and check the GPU

In [ ]:
%%capture
# Unsloth's own Colab recipe (Qwen3 (4B) Instruct notebook), with every version pinned.
# torch and triton stay as Colab ships them (unsloth 2026.10.3 needs torch < 2.15).
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps "unsloth_zoo==2026.10.3" "bitsandbytes==0.50.2" "accelerate==1.15.0" "peft==0.21.2" "unsloth==2026.10.3"
!pip install --no-deps "torchao==0.18.0"
!pip install "transformers==4.57.6"
!pip install --no-deps "trl==0.22.2"

In [ ]:
import importlib.metadata as md, torch
for name in ['torch', 'triton', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets', 'torchao']:
    try:
        print(f'{name:14s} {md.version(name)}')
    except md.PackageNotFoundError:
        print(f'{name:14s} not installed')
print('CUDA', torch.version.cuda, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: set Runtime -> T4 GPU')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

## 2. Google Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, glob, shutil, hashlib, random

DRIVE = '/content/drive/MyDrive/clarity'
TRAIN_FILE = f'{DRIVE}/data/sft_train.jsonl'
DEV_FILE = f'{DRIVE}/data/sft_dev.jsonl'
OUT = f'{DRIVE}/out'
CHECKPOINTS = f'{OUT}/checkpoints'
ADAPTER_DIR = f'{OUT}/lora_adapter'
GGUF_NAME = 'clarity-simplify-q4_k_m.gguf'
MERGED_DIR = f'{OUT}/merged_16bit'

BASE_MODEL = 'unsloth/Qwen3-4B-Instruct-2507'
MAX_SEQ_LEN = 1024
SEED = 42
LORA = dict(r=16, lora_alpha=16, lora_dropout=0,
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
TRAIN = dict(epochs=1, lr=2e-4, scheduler='cosine', warmup_ratio=0.03, batch=2, grad_accum=8,
             eval_steps=100, save_steps=100)

for f in (TRAIN_FILE, DEV_FILE):
    assert os.path.exists(f), f'Missing {f}: upload it to MyDrive/clarity/data/ (see training/README.md)'
os.makedirs(CHECKPOINTS, exist_ok=True)
random.seed(SEED)
print('Train file sha256', hashlib.sha256(open(TRAIN_FILE, 'rb').read()).hexdigest())
print('Dev file sha256  ', hashlib.sha256(open(DEV_FILE, 'rb').read()).hexdigest())
print('Compare with training/data/STATS.md.')

## 3. Base model in 4-bit and the LoRA adapter

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA['r'],
    lora_alpha=LORA['lora_alpha'],
    lora_dropout=LORA['lora_dropout'],
    target_modules=LORA['target_modules'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
model.print_trainable_parameters()

## 4. Chat template and data

The training text must be **token-for-token what Ollama sends**, then the answer. Ollama's Qwen3 template (copied into `training/Modelfile.template`) renders the system prompt followed by a blank line before `<|im_end|>`, and opens the answer with a bare `<|im_start|>assistant\n`. The Hugging Face template of this checkpoint instead inserts an empty `<think>\n\n</think>\n\n` before the answer (checked on 2026-10-20), which would teach the model to print empty think tags under Ollama. So each training text is built exactly as Ollama renders it (`ollama_prompt` + answer + `<|im_end|>`); the `qwen3-instruct` template is still set on the tokenizer, and the cell prints how the two renderings differ for the record. The JSONL files keep the app's exact prompts.

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template='qwen3-instruct')

def ollama_prompt(messages):
    """What Ollama's template renders for a system + user chat (see Modelfile.template)."""
    system = next(m['content'] for m in messages if m['role'] == 'system')
    user = next(m['content'] for m in messages if m['role'] == 'user')
    return f'<|im_start|>system\n{system}\n\n<|im_end|>\n<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n'

def to_text(row):
    messages = row['messages']
    return {'text': ollama_prompt(messages) + messages[2]['content'] + '<|im_end|>\n'}

raw = load_dataset('json', data_files={'train': TRAIN_FILE, 'dev': DEV_FILE})
data = raw.map(to_text)

example = raw['train'][0]['messages']
rendered = data['train'][0]['text']
hf = tokenizer.apply_chat_template(
    [{**m, 'content': m['content'] + '\n\n'} if m['role'] == 'system' else m for m in example],
    tokenize=False, add_generation_prompt=False)
print('qwen3-instruct template renders the same text:', hf == rendered)
if hf != rendered:
    i = next((k for k, (a, b) in enumerate(zip(hf, rendered)) if a != b), min(len(hf), len(rendered)))
    print('  first difference at char', i, '| template:', repr(hf[i:i + 40]), '| Ollama:', repr(rendered[i:i + 40]))
assert rendered.startswith('<|im_start|>system\n') and rendered.endswith(example[2]['content'] + '<|im_end|>\n')
print('\nOne training example exactly as Ollama will see it (plus the answer):\n')
print(rendered)

In [ ]:
def token_length(row):
    return {'n_tokens': len(tokenizer(row['text'], add_special_tokens=False)['input_ids'])}

data = data.map(token_length)
for split in ('train', 'dev'):
    lengths = sorted(data[split]['n_tokens'])
    too_long = sum(n > MAX_SEQ_LEN for n in lengths)
    print(f'{split}: {len(lengths)} examples, tokens median {lengths[len(lengths)//2]}, '
          f'p95 {lengths[int(0.95*(len(lengths)-1))]}, max {lengths[-1]}, over {MAX_SEQ_LEN}: {too_long} (dropped)')
data = data.filter(lambda row: row['n_tokens'] <= MAX_SEQ_LEN)

## 5. Trainer (loss on the assistant answer only)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data['train'],
    eval_dataset=data['dev'],
    args=SFTConfig(
        dataset_text_field='text',
        max_length=MAX_SEQ_LEN,
        packing=False,
        num_train_epochs=TRAIN['epochs'],
        per_device_train_batch_size=TRAIN['batch'],
        per_device_eval_batch_size=TRAIN['batch'],
        gradient_accumulation_steps=TRAIN['grad_accum'],
        learning_rate=TRAIN['lr'],
        lr_scheduler_type=TRAIN['scheduler'],
        warmup_ratio=TRAIN['warmup_ratio'],
        fp16=not is_bfloat16_supported(),  # a T4 has no bf16
        bf16=is_bfloat16_supported(),
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=TRAIN['eval_steps'],
        save_strategy='steps',
        save_steps=TRAIN['save_steps'],
        save_total_limit=3,
        output_dir=CHECKPOINTS,
        seed=SEED,
        data_seed=SEED,
        report_to='none',
        dataset_num_proc=2,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Masking check: the only trained tokens of example 0 must be its answer
labels = trainer.train_dataset[0]['labels']
trained = tokenizer.decode([t for t in labels if t != -100])
answer = data['train'][0]['messages'][2]['content']
print('Trained tokens of example 0:\n', repr(trained))
assert trained.strip().startswith(answer.strip()[:50]), 'Masking looks wrong: the trained tokens are not the assistant answer'

## 6. Train (resumes from the last Drive checkpoint)

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last = get_last_checkpoint(CHECKPOINTS) if os.path.isdir(CHECKPOINTS) else None
print('Resuming from', last) if last else print('Starting a new run')
stats = trainer.train(resume_from_checkpoint=last)
print(stats)

with open(f'{OUT}/log_history.json', 'w') as f:
    json.dump(trainer.state.log_history, f, indent=1)
evals = [h for h in trainer.state.log_history if 'eval_loss' in h]
print('Dev loss by step:', [(h['step'], round(h['eval_loss'], 4)) for h in evals])

## 7. Sanity check: 5 dev examples (greedy)

In [ ]:
FastLanguageModel.for_inference(model)
rng = random.Random(SEED)
for i in rng.sample(range(len(raw['dev'])), 5):
    messages = raw['dev'][i]['messages']
    prompt = ollama_prompt(messages)
    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
    output = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    meta = raw['dev'][i]['meta']
    print('=' * 100)
    print(f"Grade {meta['g']} ({meta['kind']}), source FKGL {meta['fkgl_src']}, target FKGL {meta['fkgl_tgt']}")
    print('SOURCE:', messages[1]['content'].split('TEXT TO REWRITE:\n')[-1])
    print('TARGET:', messages[2]['content'])
    print('OUTPUT:', output)

## 8. Save the LoRA adapter to Drive

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Adapter saved to', ADAPTER_DIR)

## 9. Export GGUF q4_k_m for Ollama

Merges the adapter and quantizes with llama.cpp (takes 10–20 minutes). The file lands in Drive as `MyDrive/clarity/out/clarity-simplify-q4_k_m.gguf`. If this cell runs out of RAM, skip to step 10.

In [ ]:
LOCAL_GGUF = '/content/gguf'
model.save_pretrained_gguf(LOCAL_GGUF, tokenizer, quantization_method='q4_k_m')
found = sorted(glob.glob(f'{LOCAL_GGUF}/**/*.gguf', recursive=True) + glob.glob(f'{LOCAL_GGUF}*.gguf'))
found = [f for f in found if 'q4_k_m' in f.lower()] or found
assert found, 'No GGUF file was written; use step 10 instead'
print('Written:', found)
shutil.copy(found[0], f'{OUT}/{GGUF_NAME}')
digest = hashlib.sha256()
with open(f'{OUT}/{GGUF_NAME}', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        digest.update(chunk)
print(f'{OUT}/{GGUF_NAME}', os.path.getsize(f'{OUT}/{GGUF_NAME}') // 2**20, 'MB, sha256', digest.hexdigest())

## 10. Fallback only: merged 16-bit weights

Run this **only if step 9 failed** (usually out of RAM). It saves the merged model (about 8 GB) to `MyDrive/clarity/out/merged_16bit/`; `training/README.md` explains converting it to GGUF on your PC with llama.cpp.

In [ ]:
RUN_FALLBACK = False  # set True only if step 9 failed
if RUN_FALLBACK:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method='merged_16bit')
    print('Merged 16-bit model saved to', MERGED_DIR)
else:
    print('Skipped (step 9 succeeded)')

## 11. Run record

In [ ]:
import importlib.metadata as md
record = {
    'base_model': BASE_MODEL, 'max_seq_len': MAX_SEQ_LEN, 'seed': SEED, 'lora': LORA, 'train': TRAIN,
    'train_examples': len(data['train']), 'dev_examples': len(data['dev']),
    'global_steps': trainer.state.global_step,
    'final_dev_loss': next((h['eval_loss'] for h in reversed(trainer.state.log_history) if 'eval_loss' in h), None),
    'gpu': torch.cuda.get_device_name(0),
    'versions': {n: md.version(n) for n in ['torch', 'unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'bitsandbytes']},
}
with open(f'{OUT}/run_record.json', 'w') as f:
    json.dump(record, f, indent=1)
print(json.dumps(record, indent=1))